# Notebook 05 — MHA, MQA, GQA, FlashAttention, and KV Caches

    ## Learning objectives

    - Compare multi-head, multi-query, and grouped-query attention
- Separate exact attention algorithms from approximation
- Estimate KV-cache memory and understand FlashAttention's IO benefit

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 5.1 Sharing K/V heads

Multi-head attention (MHA) has one K/V head per query head. Multi-query attention
(MQA) shares one K/V head across all query heads. Grouped-query attention (GQA) uses
an intermediate number of K/V heads. During decoding, fewer K/V heads reduce cache
memory and memory bandwidth, often with a smaller quality tradeoff than MQA.


In [ ]:
def kv_cache_gib(layers, kv_heads, head_dim, tokens, batch=1, bytes_per_value=2):
    # factor 2 stores both K and V
    return 2 * layers * kv_heads * head_dim * tokens * batch * bytes_per_value / 2**30

for name, kv_heads in {"MHA": 32, "GQA": 8, "MQA": 1}.items():
    size = kv_cache_gib(layers=32, kv_heads=kv_heads, head_dim=128,
                         tokens=32_768, batch=1)
    print(f"{name}: {size:.2f} GiB")


## 5.2 FlashAttention

Standard attention conceptually computes \(S=QK^T\), softmaxes rows, then multiplies
by V. The arithmetic remains quadratic in sequence length, but writing the full score
and probability matrices to high-bandwidth memory is expensive. FlashAttention tiles
the computation, maintains online softmax statistics, and recomputes selected values
during backward. It is **exact attention up to numerical precision**, not sparse or
linear attention. Its main win is IO and intermediate-memory reduction.


In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention

q = k = v = torch.randn(2, 4, 128, 64)
# PyTorch dispatches to an eligible fused backend for the hardware/dtype/shape.
out = scaled_dot_product_attention(q, k, v, is_causal=True)
print(out.shape)
if torch.cuda.is_available():
    print(torch.backends.cuda.sdp_kernel())
else:
    print("CPU/MPS demonstration: fused CUDA FlashAttention is not expected here.")


## 5.3 Prefill versus decode

Prefill processes the prompt in parallel and is compute-heavy. Decode generates one
token per sequence step, reads the growing KV cache, and is often memory-bandwidth
limited. Prefix caching reuses shared prompt KV states. Paged/block-based caches reduce
fragmentation. Quantized/offloaded caches trade precision or transfer cost for capacity.
Always report prompt length, output length, batch/concurrency, TTFT, and tokens/second.


## 5.4 Attention variants in shape notation

Let query heads be \(H_q\), KV heads \(H_{kv}\), and group size
\(g=H_q/H_{kv}\). MHA has \(H_{kv}=H_q\), MQA has \(H_{kv}=1\), and GQA lies between.
During attention, each K/V head is logically shared by g query heads. The Q projection
remains \(D\times D\); K and V projections shrink in proportion to KV heads. Output shape
remains `[B,T,D]`, so downstream blocks are unchanged.

Other efficiency families solve different problems. Sliding-window attention restricts each
token to a local neighborhood, reducing long-sequence work but limiting direct interaction.
Block-sparse attention chooses structured connections. Linear-attention methods replace or
reorder softmax attention using kernel/state formulations and are generally approximate or
architecturally different. Mixture-of-experts sparsifies MLP parameter activation, not
attention. Do not group all of these under “FlashAttention”: Flash changes execution of the
same dense attention result.


In [ ]:
# Compare projection parameters and cache bytes for MHA/GQA/MQA.
def attention_accounting(width, q_heads, kv_heads, layers, tokens, dtype_bytes=2):
    head_dim = width // q_heads
    q = width * width
    k_and_v = 2 * width * (kv_heads * head_dim)
    out = width * width
    cache = 2 * layers * kv_heads * head_dim * tokens * dtype_bytes
    return q + k_and_v + out, cache

for label, kv in [("MHA", 32), ("GQA-8", 8), ("GQA-4", 4), ("MQA", 1)]:
    params, cache = attention_accounting(4096, 32, kv, 32, 32_768)
    print(f"{label:6} projections={params/1e6:7.1f}M  cache={cache/2**30:5.2f} GiB")


## 5.5 Online softmax: why tiling can be exact

Softmax seems to require an entire row because its denominator sums all keys. Online
softmax processes blocks while maintaining a running maximum \(m\) and normalized sum
\(l\). When a new block has a larger maximum, earlier accumulators are rescaled by
\(e^{m_{old}-m_{new}}\). The output accumulator is updated with the same correction. This
lets a kernel tile Q/K/V in fast on-chip memory without storing the full score matrix in
device memory. Backward recomputes inexpensive intermediates rather than reading huge saved
matrices.

Kernel dispatch has constraints: device generation, dtype, head dimension, mask type,
dropout, contiguity, and library version. PyTorch scaled-dot-product attention selects among
Flash, memory-efficient, and math backends when eligible. A fallback is correct but can have
very different memory/performance. Benchmark after warmup with synchronization, realistic
shapes, and peak-memory measurement.


In [ ]:
# Simple backend-agnostic attention benchmark scaffold.
import time
def benchmark_sdpa(B=2, H=8, T=512, Dh=64, repeats=10, device="cpu"):
    q = torch.randn(B, H, T, Dh, device=device)
    for _ in range(2):
        torch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True)
    if device == "cuda": torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        torch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True)
    if device == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats

device = "cuda" if torch.cuda.is_available() else "cpu"
for T in [128, 256, 512]:
    print(T, f"{benchmark_sdpa(T=T, repeats=3, device=device)*1000:.2f} ms")


## 5.6 KV-cache operations reference

The cache stores per-layer K/V after positional transformation. Dynamic caches grow with
tokens and are convenient; static caches preallocate a maximum shape and can work better
with compilation but may waste space. Sliding caches retain a fixed recent window. Offloaded
caches move data across slower links. Quantized caches reduce bytes but add conversion and
may affect quality. Prefix caching reuses identical prompt blocks across requests; it needs
canonical token sequences and cache-aware scheduling.

Continuous batching interleaves decode steps for active sequences. Paged allocation maps
logical token blocks to physical cache blocks, reducing fragmentation and enabling sharing.
Capacity planning must include batch/concurrency and total cached tokens, not only maximum
context per request. Monitor cache utilization, evictions, prefix hit rate, prefill/decode
throughput, queue time, TTFT, and inter-token latency. OOM under load is an admission-control
failure even if one maximum-length request fits in isolation.


## 5.7 Efficient-attention decision table

| Technique | Changes mathematical attention? | Main benefit | Main tradeoff |
|---|---:|---|---|
| FlashAttention/SDPA fused kernel | No | Less HBM IO/intermediate memory | Hardware/shape/kernel constraints |
| GQA/MQA | Architecture changes K/V heads | Smaller cache and decode bandwidth | Must be trained/conversion-quality tested |
| Sliding window | Yes, restricts connections | Lower long-context work/cache | Loses direct distant attention |
| Prefix cache | No model change | Reuses identical prompt KV | Hit rate, memory, routing complexity |
| Quantized/offloaded KV | Numeric/storage change | More cached tokens | Conversion, quality, transfer latency |
| Speculative decoding | Same target distribution if implemented exactly | Faster decode with accepted drafts | Draft cost and acceptance dependence |

Never report “attention speed” without batch, Q/K lengths, heads, head dimension, dtype, causal/mask,
device, backend, warmup, forward/backward, and synchronization. Training and decode are different
benchmarks. A memory-efficient kernel may allow a larger batch that improves throughput even if a
single operation is not much faster.

When serving, capacity is total active cached tokens across sequences. Apply admission control before
the allocator reaches failure; leave workspace/deployment headroom; monitor rather than infer cache
health from GPU utilization alone.


## Exercises

    1. Calculate cache memory for your chosen model at three context lengths.
2. Benchmark PyTorch attention with and without a fused backend on suitable hardware.
3. Explain why FlashAttention does not remove quadratic compute complexity.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
